
# PS2: Mini-batch SGD and CNN

Advanced Learning 2026/7.



---


Imports:

In [1]:
import numpy as np # You are allowed to use  only numpy.
import time, sys, cProfile
import matplotlib.pyplot as plt

Please import the objects in `srcPS2.py` file into the workspace

In [2]:
try:  # Colab: mount Drive. Elsewhere: srcPS2.py is expected next to the notebook.
    from google.colab import drive
    drive.mount("/content/drive/")
except ImportError:
    pass

In [3]:
your_drive_path_to_src_file = "/content/drive/MyDrive/[PATH TO YOUR DIRECTORY WHERE SRC IS IN]]"
sys.path.append(your_drive_path_to_src_file)

In [4]:
from srcPS2 import *

Note that now all of the network's objects from PS1 are loaded into the workspace.

**Alternatively**, you can upload srcPS2.py in the files tab found on the left hand side without needing to mount your drive, and import as is.

---

### 0. Loading data

As in PS1, you will evaluate your home-made network on the `mnist` dataset.   
The MNIST dataset is a large dataset of handwritten digits that is commonly used for training various image and vision models.  



In [5]:
from keras.datasets import mnist
from keras.utils import to_categorical
# load MNIST from server
# Using a standard library (keras.datasets) to load the mnist data
(x_train, y_train), (x_test, y_test) = mnist.load_data()

---

## PS2 Part 1: Mini-batch SGD



**Welcome back**.   






Mini-batch Stochastic Gradient Descent (Mini-batch SGD) is an optimization algorithm used to train machine learning models, particularly neural networks. It strikes a balance between two other methods:

    Batch Gradient Descent – Uses the entire dataset to compute gradients and update the model parameters.
        ✅ Accurate gradients
        ❌ Slow and memory-intensive for large datasets

    Stochastic Gradient Descent (SGD) – Uses a single data point to compute gradients for each update.
        ✅ Fast and efficient
        ❌ Noisy updates, which can lead to instability

  
**Mini-batch SGD**.

Mini-batch SGD computes gradients and updates the model parameters using small, randomly selected batches of data (typically 32, 64, or 128 samples).

    ✅ Faster than batch gradient descent
    ✅ Less noisy than pure SGD
    ✅ Efficient use of memory
    ✅ Enables vectorization for faster computation on GPUs

**How it Works**.

    Shuffle the dataset to avoid bias.

    Divide the data into small batches (e.g., 64 samples per batch).

    For each epoch, iterate through the mini-batches:
        Perform forward pass to calculate predictions.
        Compute the loss for the mini-batch.
        Perform backpropagation to compute gradients.
        Update model parameters using the gradients.

    Repeat until the model converges.

**Algorithm**

For each mini-batch $B={x_1,x_2,...,x_m}B={x_1​,x_2​,...,x_m​}$ of size $m$, update the weights $w$ as:    

$$
w=w−η⋅\frac{1}{m}∑_{i=1}^m ∇L(w,x_i)
$$

Where:   
*  $η$ = learning rate.
* $∇L(w,x_i)$ = gradient of the loss for sample $x_i$.   

Advantages

    Efficient and scalable – Works well with large datasets.
    Smooth convergence – Less noisy than SGD but faster than full batch training.
    Parallelization – Batches can be processed in parallel on GPUs.

Disadvantages

    Choosing batch size can affect performance. Small batches lead to noise, large batches consume more memory.
    Convergence may be slower than batch gradient descent for some problems.


### Mini-batch Questions.
**P1Q1**:  

Please answer the following questions providing statistical justifications and mathematical formulas when needed.

P1Q1a: Explain how the mini-batch size impacts convergence, model performance, and computational efficiency.



**Answer (P1Q1a).** The mini-batch gradient $g_B=\frac1m\sum_{i\in B}\nabla L(w,x_i)$ is an unbiased estimate of the full gradient, and its variance is about $\Sigma/m$, where $\Sigma$ is the variance of a single-sample gradient. So the noise (std) goes down like $1/\sqrt m$.

* **Convergence:** bigger batches give smoother steps and allow a bigger learning rate. But there are only $N/m$ updates per epoch, so doubling $m$ halves the number of steps while the noise only goes down by $\sqrt2$. Per epoch, large batches make less progress, and after some size a bigger batch doesn't really help anymore.
* **Performance:** the noise of small batches works as a regularizer, it pushes towards flatter minima that usually generalize better. Very large batches tend to generalize a bit worse unless the learning rate is retuned. Very small batches ($m\approx1$) are just noisy and unstable.
* **Efficiency:** a batch is one matrix-matrix product instead of $m$ matrix-vector products, so the time per update grows much slower than $m$ (until the CPU/GPU is saturated). Memory grows linearly with $m$. In practice something between 32 and 256 is a good compromise.

**P1Q1b:**   
Why does mini-batch SGD require a learning rate schedule, and what are the common scheduling strategies?



**Answer (P1Q1b).** With a constant learning rate the stochastic gradient is not zero even at the minimum (its variance is still about $\Sigma/m$), so SGD keeps bouncing around the minimum, in a region that gets bigger with $\eta$. A big $\eta$ is good at the start (fast progress) and a small one at the end (less noise). To really converge we need a decaying step, for example with the Robbins–Monro conditions $\sum_t\eta_t=\infty$ and $\sum_t\eta_t^2<\infty$. At the very beginning a big learning rate can also blow up, which is why a short warm-up is often used.

Common schedules: step decay $\eta_0\gamma^{\lfloor t/s\rfloor}$, exponential decay, $1/t$ decay, cosine annealing $\eta_t=\eta_{\min}+\frac12(\eta_0-\eta_{\min})(1+\cos(\pi t/T))$, linear warm-up, one-cycle, and reduce-on-plateau (lower the lr when the validation loss stops improving).
Below I use a 1-epoch warm-up followed by cosine annealing (`WarmupCosineLR`).

P1Q1c: What role does batch normalization play in mini-batch SGD, and why is it essential for deep neural networks?



**Answer (P1Q1c).** BN normalizes each feature with the mean and variance of the current mini-batch, then rescales it with learned parameters:
$$\hat x_i=\frac{x_i-\mu_B}{\sqrt{\sigma_B^2+\varepsilon}},\qquad y_i=\gamma\hat x_i+\beta .$$
* It keeps the activations of every layer at a similar scale, so the gradients don't explode or vanish when the network gets deeper, and training depends much less on the initialization. This is why it matters so much for deep networks.
* It makes the loss smoother and the layer insensitive to the scale of its weights, so larger learning rates can be used and training is faster.
* $\mu_B$ and $\sigma_B$ are computed on a random batch, so they are noisy, which adds a small regularization effect (a bit like dropout).
* It depends on the batch size: with very small batches the statistics are too noisy. At test time the running averages from training are used instead of batch statistics.

P1Q1d: How does mini-batch SGD differ in performance compared to full-batch gradient descent when dealing with non-convex optimization problems?



**Answer (P1Q1d).** Full-batch GD is deterministic: on a non-convex loss it goes to the nearest critical point and stays there. Near saddle points, which are very common in high dimension, the gradient is almost zero and GD becomes very slow. Each step also costs a full pass over the data.

Mini-batch SGD follows the true gradient plus a noise with variance about $\Sigma/m$. This noise actually helps: it lets SGD escape saddle points and sharp minima, and it tends to end in wide minima that generalize better. It also makes $N/m$ updates per epoch instead of one, so it is much cheaper for the same progress. The downside is that it doesn't converge exactly without a decaying learning rate. In practice, on neural networks, mini-batch SGD reaches the same or a better test error than full-batch GD with much less compute.

P1Q1e: How does mini-batch SGD perform when training on imbalanced datasets, and what strategies can mitigate its challenges?



**Answer (P1Q1e).** The batch gradient is roughly an average of the per-class gradients weighted by the class frequencies, so the majority classes dominate it. A class with frequency $\pi$ is missing from a batch with probability $(1-\pi)^m$: for $\pi=1\%$ and $m=32$ this is $0.99^{32}\approx72\%$, so most updates don't see that class at all. The model ends up biased towards the majority, with a good accuracy but a poor recall on the small classes.

What helps:
* weight the loss of class $k$ by about $1/\pi_k$;
* oversample the minority / undersample the majority, or build balanced batches;
* a loss like the focal loss;
* data augmentation for the minority classes;
* evaluate with balanced accuracy, F1 or PR-AUC instead of accuracy, and adjust the decision threshold.

P1Q1f: Explain why larger mini-batch sizes are preferred during inference but not necessarily during training.


**Answer (P1Q1f).** At inference there is no gradient and no update, and the prediction for a sample doesn't depend on the batch size (BN uses its running statistics). So the batch size only changes the speed: bigger batches use the hardware better (matrix-matrix products, less overhead per call), and memory is less of a problem because no activations are stored for a backward pass. The only cost is latency when requests come one at a time.

In training, the batch size changes the optimization itself: the gradient noise ($\propto1/m$), the number of updates per epoch, the implicit regularization, the BN statistics and the right learning rate. A batch that is too large means fewer useful updates and often worse generalization. So in training it is a hyperparameter to tune, while in inference it is just an engineering choice.

### Applying mini-batch SGD

Preparing the data:

In [6]:
# training data : 60000 samples
# reshape and normalize input data
x_train = x_train.reshape(x_train.shape[0], 1, 28*28)
x_train = x_train.astype('float32')
x_train /= 255
# One-hot encoding of the output.
# Currently a number in range [0,9]; Change into a vector of size 10
# e.g. number 3 will become [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
y_train = to_categorical(y_train)
# same for test data : 10000 samples
x_test = x_test.reshape(x_test.shape[0], 1, 28*28)
x_test = x_test.astype('float32')
x_test /= 255
y_test = to_categorical(y_test)

P1Q2a:  

Add a new function to the class `MyNetwork` named `fit_mini_batch`. This function should be similar to `fit`, just with a mini-batch SGD implementation. You can also make other code changes to the scripts in `srcPS2` if you think they are needed. In addition, add a suitable learning rate scheduler of your choice.


YOU SHOULD (a) add the missing script to the srcPS2 code, and (b) make a copy of the function  below for review:

In [7]:
# Copy for review -- the live versions are in srcPS2.py (MyNetwork.fit_mini_batch + schedulers)

# Learning-rate schedulers: callables mapping epoch index (0-based) -> learning rate
class StepDecayLR:
    """lr = lr0 * gamma ** floor(epoch / step_size)"""
    def __init__(self, lr0, step_size=3, gamma=0.5):
        self.lr0, self.step_size, self.gamma = lr0, step_size, gamma

    def __call__(self, epoch):
        return self.lr0 * self.gamma ** (epoch // self.step_size)


class WarmupCosineLR:
    """Linear warm-up for `warmup` epochs, then cosine annealing from lr0 to lr_min."""
    def __init__(self, lr0, total_epochs, warmup=1, lr_min=0.0):
        self.lr0, self.T, self.warmup, self.lr_min = lr0, total_epochs, warmup, lr_min

    def __call__(self, epoch):
        if epoch < self.warmup:
            return self.lr0 * (epoch + 1) / self.warmup
        t = (epoch - self.warmup) / max(1, self.T - self.warmup)
        return self.lr_min + 0.5 * (self.lr0 - self.lr_min) * (1 + np.cos(np.pi * t))


def fit_mini_batch(self, x_train, y_train, batch_size, epochs, learning_rate):
    """
    Trains the network using mini-batch gradient descent.

    Parameters:
      x_train : np.array
          Training data.
      y_train : np.array
          Training labels.
      batch_size : int
          Size of each mini-batch.
      epochs : int
          Number of training epochs.
      learning_rate : float
          Learning rate for updating the weights.
    """
    samples = len(x_train)
    n_batches = int(np.ceil(samples / batch_size))
    # flatten each sample: (N, 1, d) -> (N, d) so a batch is a (B, d) matrix
    X = x_train.reshape(samples, -1)
    Y = y_train.reshape(samples, -1)
    print("Training on {:,} samples, batch size {}, {} batches/epoch:".format(
        samples, batch_size, n_batches))
    for i in range(epochs):
        # learning_rate is either a constant or a scheduler: callable(epoch) -> lr
        lr = learning_rate(i) if callable(learning_rate) else learning_rate
        perm = np.random.permutation(samples)          # reshuffle every epoch
        err = 0
        for b in range(n_batches):
            idx = perm[b * batch_size:(b + 1) * batch_size]
            output, y_batch = X[idx], Y[idx]
            for layer in self.layers:
                output = layer.forward_propagation(output)
            # mse / mse_grad divide by y.size = B*n_out, so the gradient is
            # already the batch MEAN: w <- w - lr * (1/m) * sum_i grad L_i
            err += self.loss(y_batch, output) * len(idx)
            grad = self.loss_grad(y_batch, output)
            for layer in reversed(self.layers):
                grad = layer.backward_propagation(grad, lr)
        err /= samples
        print("Training epoch %d/%d   lr=%.4f   error=%f" % (i + 1, epochs, lr, err))


    Test the two networks below to compare the differences between GD and mini-batch SGD:

In [8]:
#### GD implementaiton ####

# Network Architecture
np.random.seed(0)
netGD = MyNetwork()

netGD.add(Affine_Layer(28*28, 128))
netGD.add(ActivationLayer(tanh, tanh_grad))
netGD.add(Affine_Layer(128, 64))
netGD.add(ActivationLayer(tanh, tanh_grad))
netGD.add(Affine_Layer(64, 10))
netGD.add(ActivationLayer(tanh, tanh_grad))

netGD.use_loss(mse, mse_grad)

epoch_num = 10
lr = 0.05
t1 = time.time()
netGD.fit(x_train[:10000], y_train[:10000], epochs=epoch_num, learning_rate=lr)
print(f"Total process time: {round(time.time() - t1,3)}")

Training on 10,000 samples:


Training epoch 1/10   error=0.042086


Training epoch 2/10   error=0.024651


Training epoch 3/10   error=0.019315


Training epoch 4/10   error=0.016296


Training epoch 5/10   error=0.014276


Training epoch 6/10   error=0.012772


Training epoch 7/10   error=0.011572


Training epoch 8/10   error=0.010579


Training epoch 9/10   error=0.009733


Training epoch 10/10   error=0.008999
Total process time: 31.64


In [9]:
output_GD = netGD.predict(x_test ,y_test )


Test set: Avg. loss: 0.0124, Accuracy: 9391/10000 (94%)



In [10]:
#### mini-batch SGD implementaiton ####

# Network Architecture
np.random.seed(0)
netMiniGD = MyNetwork()

netMiniGD.add(Affine_Layer(28*28, 128))
netMiniGD.add(ActivationLayer(tanh, tanh_grad))
netMiniGD.add(Affine_Layer(128, 64))
netMiniGD.add(ActivationLayer(tanh, tanh_grad))
netMiniGD.add(Affine_Layer(64, 10))
netMiniGD.add(ActivationLayer(tanh, tanh_grad))

netMiniGD.use_loss(mse, mse_grad)

epoch_num = 10
# warm-up 1 epoch + cosine decay. lr0=3 (0.05*128 = 6.4 diverged)
lr_sched = WarmupCosineLR(lr0=3.0, total_epochs=epoch_num, warmup=1)
t2 = time.time()
netMiniGD.fit_mini_batch(x_train[:10000], y_train[:10000], batch_size=128, epochs=epoch_num, learning_rate=lr_sched)
print(f"Total process time: {round(time.time() - t2,3)}")


Training on 10,000 samples, batch size 128, 79 batches/epoch:


Training epoch 1/10   lr=3.0000   error=0.418738


Training epoch 2/10   lr=3.0000   error=0.047687


Training epoch 3/10   lr=2.9095   error=0.038300


Training epoch 4/10   lr=2.6491   error=0.032507


Training epoch 5/10   lr=2.2500   error=0.027469


Training epoch 6/10   lr=1.7605   error=0.023977


Training epoch 7/10   lr=1.2395   error=0.022133


Training epoch 8/10   lr=0.7500   error=0.020900


Training epoch 9/10   lr=0.3509   error=0.020365


Training epoch 10/10   lr=0.0905   error=0.020004
Total process time: 2.055


In [11]:
outputMiniGSD = netMiniGD.predict(x_test ,y_test )


Test set: Avg. loss: 0.0223, Accuracy: 9185/10000 (92%)



P1Q2b:   
Please answer the following:
* Which implementation was faster?
* with lower training error?
* with better test accuracy?    

and explain how these results fit your answers in Q1A.

**Answer (P1Q2b).** Same architecture, same initialization (`np.random.seed(0)`), 10,000 training samples, 10 epochs:

| | `fit` (one sample per update) | `fit_mini_batch` ($m=128$, warm-up + cosine, $\eta_0=3$) |
|---|---|---|
| Updates per epoch | 10,000 | 79 |
| Training time | 31.6 s | **2.1 s** |
| Final training error (MSE) | **0.0090** | 0.0200 |
| Test accuracy | **93.9 %** | 91.9 % |

* **Faster: mini-batch, about 15×.** Both do the same number of flops per epoch, but `fit` does 10,000 small forward/backward passes in Python, while `fit_mini_batch` does 79 passes with big matrix products that numpy runs very efficiently. This is the efficiency point from Q1a.
* **Lower training error and better test accuracy: `fit`.** In 10 epochs it makes 100,000 updates against 790 for the mini-batch. Averaging 128 gradients reduces the noise, but it also divides the number of steps by 128, so per epoch the mini-batch makes less progress (Q1a again). A bigger learning rate only partly compensates: $0.05\times128=6.4$ diverged (error around 1, the tanh units saturate), 3 was about the largest stable value, and with a constant 0.05 the mini-batch only got 79 % test accuracy.
* At equal time instead of equal epochs, the mini-batch could run about 150 epochs in the 31.6 s used by `fit`, so per second of compute it is clearly better.
* The test accuracy follows the training error and neither model overfits here, so the difference comes from the optimization (number of updates), not from generalization.

## PS2 Part 2: CNN



A Convolutional Neural Network (CNN) is a type of a deep neural network that can perform well at image recognition and analysis tasks.


> In this problem set we will add CNN compatibility to our numpy-based neural network from PS 1.

The main components needed to implement a CNN architecture:  

**Convolution layer:** In the convolutional layers, filters are applied to the image to identify patterns and features. Imagine a sliding window that moves across the image, detecting edges, shapes, and other building blocks.  

**Pooling layer:** Pooling layers downsample the data, reducing its complexity and computational cost. This helps prevent overfitting, where the model memorizes training data instead of learning generalizable patterns.  

**Reshape layer:**
Reshaping the output of the convolution and pooling layers into the fully connected layer.   


A USEFUL HINT:

*Keeping account of the input and output shapes along the CNN network is crucial for the model to work as intended*



Useful resource:  
* [CS231](https://cs231n.github.io/convolutional-networks/).
* [Convolutional Neural Network From Scratch](https://www.kaggle.com/code/lusfernandotorres/convolutional-neural-network-from-scratch).
* [CNN with PyTorch](https://www.kaggle.com/code/sdelecourt/cnn-with-pytorch-for-mnist)

#### Data transformations





(Here I added a reshape that I originally intended for you to apply at model time).

In [12]:
from keras.datasets import mnist
from keras.utils import to_categorical
# load MNIST from server
# Using a standard library (keras.datasets) to load the mnist data
(x_train, y_train), (x_test, y_test) = mnist.load_data()

In [13]:
# training data : 60000 samples
# reshape and normalize input data
x_train = x_train.astype('float32')
x_train /= 255
x_train = x_train.reshape(x_train.shape[0], 1, x_train.shape[1],x_train.shape[2])
# One-hot encoding of the output.
# Currently a number in range [0,9]; Change into a vector of size 10
# e.g. number 3 will become [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
y_train = to_categorical(y_train)
y_train = y_train.reshape(y_train.shape[0], 1, y_train.shape[1])

# same for test data : 10000 samples
x_test = x_test.astype('float32')
x_test /= 255
x_test = x_test.reshape(x_test.shape[0],1, x_test.shape[1],x_test.shape[2])
y_test = to_categorical(y_test)
y_test = y_test.reshape(y_test.shape[0],1, y_test.shape[1])

In [14]:
print("x_train shape",x_train.shape)
print("x_test shape",x_test.shape)
print("y_train shape",y_train.shape)
print("y_test shape",y_test.shape)

x_train shape (60000, 1, 28, 28)
x_test shape (10000, 1, 28, 28)
y_train shape (60000, 1, 10)
y_test shape (10000, 1, 10)


Unlike the multilayer perceptron (MLP) architecture, the input to the convolution net is not flattened. In our case, the size of a single `MNIST` datapoint is (1,28,28). The first dimension is the number of channels (e.g. RGB  for color images). During training the inputs and outputs often  change shape as they pass through the convolution, pooling, and the other layers of the network.

---

### A. Convolution Layer

Here you are asked to implement a basic Convolution layer with the following components:



1.   Layer initialization - initialize the layer's filters using Normalized Xavier/Hu weights
2.   A generator function that yields all the available sliding windows in the input image.
3.   Forward propogation for the convolution layer
4.   Backward propogation for the convolution layer

[CS231](https://cs231n.github.io/convolutional-networks/#conv) provides a great overview both mathematically and visually,   
and [CS230](https://cs230.stanford.edu/section/4/) provides a good overview about Xavier initialization.




P2Q1a:  (10pts).   
Please complete the missing code (`## SOLUTION REQUIRED!!`) so that the class works properly.    
You can deviate slightly from the recommended structure below, but please remain true to the class structure and compatibility with the rest of the network classes and functions. The idea here is to write YOUR OWN version of CNN, so please try not to google the answers.





In [15]:
class Convolution:
    # convolution layer using num_filters x num_filters filters.
    # size is the length/width of the filter window with shape (size x size)
    # In this simple implementation, there is no padding.

    def __init__(self, num_filters=3, size=3, activation=None, in_channels=1):
        self.stride = 1 # for simplicity, we keep the stride=1.
        self.size = size
        self.activation = activation
        self.last_input = None

        # Initialize the Convolution layer filters
        # Normalized Xavier (Glorot uniform): W ~ U(-a, a), a = sqrt(6 / (fan_in + fan_out))
        # fan_in = C_in*k*k (inputs feeding one output unit), fan_out = F*k*k
        fan_in, fan_out = in_channels * size * size, num_filters * size * size
        limit = np.sqrt(6.0 / (fan_in + fan_out))
        # shape (F, C_in, k, k): one k x k kernel per (filter, input channel)
        self.filters = np.random.uniform(-limit, limit, (num_filters, in_channels, size, size))


    def patch_generator(self, image):
        # This generator (using yield) returns all the  possible patches of  the input image
        # of shape  (h,w)  for each channel c, together with the relevant y,x coordinates of the patch.
        # input_data/image must be shaped as (c,y,x), where c is the channel/filter
        ch,h,w = image.shape

        for y in range(0,h-self.size+1):
            for x in range(0,w-self.size+1,self.stride):
                patch = image[:, y:y + self.size, x:x + self.size]   # (C_in, k, k)
                yield patch, y, x


    def forward_propagation(self, image):
        # Calculate the convolution forward pass using the filters together with a
        # user specified activation function.

        # If this is the first convolution layer, the input image is size (h,w) and must be expanded to (c,h,w).
        if len(image.shape)==2:
            image = np.expand_dims(image, axis=0)
        # keep track of last input for later backward propagation
        self.last_input = image
        # Initializing the output array:
        input_dimension = image.shape[1]                                        # h (= w)
        output_dimension = (input_dimension - self.size) // self.stride + 1     # h - k + 1

        # compute output dimensions after the convolution layer
        output = np.zeros((self.filters.shape[0], output_dimension, output_dimension))

        # forward pass: one pass over the patches, all filters at once
        # (equivalent to the per-filter loop, F times fewer Python iterations)
        for patch, y, x in self.patch_generator(image):
            # output[f, y, x] = sum_{c,i,j} filters[f, c, i, j] * patch[c, i, j]
            output[:, y, x] = np.tensordot(self.filters, patch, axes=3)

        # Applying activation function (this is a small cheat/simplification
        #  because the activation should be a separate layer in itself.  )
        if self.activation is not None:
            output = self.activation.forward_propagation(output)
        return output

    def backward_propagation(self, grad_out, learning_rate):
        # Calculate the SGD in-gradients of the filters and the layer.
        # Update only the filters gradients.

        # back propagate through activation (again, a bit of a cheat)
        if self.activation is not None:
            grad_out = self.activation.backward_propagation(grad_out,learning_rate)

        grad_in = np.zeros(self.last_input.shape)    # dL/d(input), shape (C_in, h, w)
        grad_filter = np.zeros(self.filters.shape)   # dL/d(filters), shape (F, C_in, k, k)

        for patch, y, x in self.patch_generator(self.last_input):
            g = grad_out[:, y, x]                                   # (F,)
            # dL/dW[f] += dL/dout[f,y,x] * patch
            grad_filter += g[:, None, None, None] * patch
            # dL/dpatch += sum_f dL/dout[f,y,x] * W[f]
            grad_in[:, y:y + self.size, x:x + self.size] += np.tensordot(g, self.filters, axes=1)

        self.filters -= learning_rate * grad_filter  # SGD update

        return grad_in    # return the loss gradient for this layer's inputs

### B. Max-pooling Layer

As with the convolution layer, here you are asked to implement a basic max-pooling layer with the following components:


1.   A generator function that yields all the available sliding windows in the input image. You can reuse the generator from (1).
2.   Forward propogation for the max-pooling layer
3.   Backward propogation for the max-pooling layer

The pooling layer is responsible for reducing the dimensionality of the input. As with the convolution layer, the pooling layer slides a filter across the entire image input to calculate the output. Note that this layer does not have any weights.  As the filter slides through the input image, it selects the pixel with the maximum value to populate the output array.

P2Q1b: (10pts).       

Please complete the missing code (`## SOLUTION REQUIRED!!`) so that the class works properly.  





In [16]:
class MaxPool:
    def __init__(self,  size=2):
        # max pooling layer

        self.last_input = None
        self.stride = 1
        self.size = size

    def patch_generator(self, image):
        # returns a generator (using yield) of all  possible patches of size size x size
        # input_data/image must be shaped as (c,y,x)
        ch,h,w = image.shape

        for c in range(ch):
          for y in range(0,h-self.size+1):
              for x in range(0,w-self.size+1,self.stride):
                  patch = image[c, y:y + self.size, x:x + self.size]
                  yield patch, c, y, x

    def forward_propagation(self, image):
        # keep track of last input for later backward propagation
        self.last_input = image

        # compute output dimensions after the max pooling
        num_channels, h_prev, w_prev = image.shape
        h_new = (h_prev - self.size) // self.stride + 1
        w_new = (w_prev - self.size) // self.stride + 1
        # hold the values of the max pooling
        output = np.zeros((num_channels, h_new, w_new))

        # Calculate the max pool on all the image patches using the generator:
        for patch, c, y, x in self.patch_generator(image):
            output[c ,y, x] = np.max(patch)
        return output

    def backward_propagation(self, grad_out, learning_rate):
        # Calculate the gradients of the last input image
        num_channels, orig_dim, *_ = self.last_input.shape
        grad_in = np.zeros(self.last_input.shape)

        # Using the generator, go over all the channels and calcualte the max-pooling input gradients
        # The gradient flows only to the argmax of each window (local derivative of max = 1 there, 0 elsewhere).
        # With stride 1 windows overlap, so a pixel can receive gradient from several windows: hence +=.
        for patch, c, y, x in self.patch_generator(self.last_input):
            (dy, dx) = np.unravel_index(np.nanargmax(patch), patch.shape)  # (row, col) inside the window
            grad_in[c, y + dy, x + dx] += grad_out[c, y, x]

        return grad_in

### C. Reshape Layer

A reshape layer in a neural network is a transformation layer that doesn't perform any computations itself (no weights).   
Its purpose is to simply change the dimensionality (shape) of the data flowing through the network.

Here's how reshape layers work:

You define a target shape for the output of the reshape layer.   
This target shape specifies the number of elements along each dimension (width, height, channels, etc.) for the new data format.

P2Q1c:  (5pts).   
Please complete the missing code (`## SOLUTION REQUIRED!!`) so that the class works properly.  





In [17]:
class Reshape:
    def __init__(self,input_shape,output_shape):
        self.input_shape=input_shape
        self.output_shape=output_shape
    def forward_propagation(self,input_data):
        return np.reshape(input_data, self.output_shape)

    def backward_propagation(self,output_gradient,learning_rate):
        # you can ignore the learning_rate
        return np.reshape(output_gradient, self.input_shape)

### Testing Your Neural Network

### Defining our main neural network architecture

Define your CNN network's architecture:

* A convolution layer with some filters `(f)` and a window size of `(w x w)`, with a `tanh` activation function.
* A max pooling layer of a window size of `(p x p)`.  
* First affine layer that takes your input and outputs 128 nodes
* `tanh` activation layer following the first affine layer
* Second affine layer that takes the first layer's input and outputs 64 nodes
* `tanh` activation layer following the second affine layer
* Third affine layer that takes your second layer's input and outputs nodes in the size of the Y labels.
* `tanh` activation layer following the last affine layer


You can compute the shapes of the convolution and max-pool layers' outputs  as a function of the input shape, the convolution sliding window size, and the number of filters for applied. ( the stride and padding are usually also used to calculate the output shapes, but in this simple implementation we set the stride to 1 with zero padding.).

P2Q2a: (5pts).        
Calculate the input and output shapes of each layer in the network below, up until the first affine layer (including).  (mathematically) explain your answer.  

**Answer (P2Q2a).** With stride $s=1$ and no padding, a sliding $k\times k$ window gives $H_{out}=\frac{H_{in}-k}{s}+1=H_{in}-k+1$ along each spatial axis.

| Layer | Input shape | Output shape | Computation |
|---|---|---|---|
| Convolution (12 filters, $3\times3$) + ReLU | $(1,28,28)$ | $(12,26,26)$ | $28-3+1=26$; one output channel per filter |
| MaxPool ($3\times3$, stride 1) | $(12,26,26)$ | $(12,24,24)$ | $26-3+1=24$; channels unchanged |
| Reshape | $(12,24,24)$ | $(1,6912)$ | $12\cdot24\cdot24=6912$ |
| Affine 1 | $(1,6912)$ | $(1,128)$ | $W\in\mathbb R^{6912\times128}$, $b\in\mathbb R^{1\times128}$, so $6912\cdot128+128=884{,}864$ parameters |

So `c = 12`, `y_new = x_new = 24`. The check below propagates one real image through the layers.

In [18]:
# Shape check: c, y_new, x_new computed from the formula, then verified on a real image
k_conv, n_filters, k_pool, H = 3, 12, 3, x_train.shape[-1]
c = n_filters
y_new = x_new = (H - k_conv + 1) - k_pool + 1
print("c, y_new, x_new =", c, y_new, x_new, "-> flattened size", c * y_new * x_new)

_conv, _pool = Convolution(num_filters=n_filters, size=k_conv, activation=ActivationLayer(relu, relu_grad)), MaxPool(size=k_pool)
_reshape, _aff = Reshape((c, y_new, x_new), (1, c * y_new * x_new)), Affine_Layer(c * y_new * x_new, 128)
s = x_train[0]; print("input        ", s.shape)
for name, L in [("conv+relu", _conv), ("maxpool", _pool), ("reshape", _reshape), ("affine 1", _aff)]:
    s = L.forward_propagation(s); print(f"{name:13s}", s.shape)

c, y_new, x_new = 12 24 24 -> flattened size 6912
input         (1, 28, 28)
conv+relu     (12, 26, 26)
maxpool       (12, 24, 24)
reshape       (1, 6912)
affine 1      (1, 128)


P2Q2b:  (5pts).   

Please complete the missing code (`## SOLUTION REQUIRED!!`) so that the network architecture works properly.  





In [19]:
#### SOLUTION REQUIRED IN THIS BLOCK (in  reshape parameters) ####

# Network Architecture
np.random.seed(0)
CNNnet = MyNetwork()
num_classes = 10
# Reshape sizes:
## SOLUTION REQUIRED!!
c = 12           # output channels = number of conv filters
x_new = 28 - 3 + 1 - 3 + 1   # conv 3x3: 28->26, maxpool 3x3 stride 1: 26->24
y_new = 28 - 3 + 1 - 3 + 1   # 24 (square images)

# Adding convolution and max-pool layers:
CNNnet.add(Convolution(num_filters=12, size=3,
                    activation=ActivationLayer(relu, relu_grad))) #
CNNnet.add(MaxPool(size=3))

# Reshaping for first affine layer:
CNNnet.add(Reshape((c,y_new,x_new),(1,c*y_new*x_new)))

# Adding affine layers and activation functions:
CNNnet.add(Affine_Layer(c*y_new*x_new, 128))
CNNnet.add(ActivationLayer(tanh, tanh_grad))
CNNnet.add(Affine_Layer(128, 64))
CNNnet.add(ActivationLayer(tanh, tanh_grad))
CNNnet.add(Affine_Layer(64, num_classes))
CNNnet.add(ActivationLayer(tanh, tanh_grad))

# CNNnet.add(ActivationLayer(relu, relu_grad))
# CNNnet.add(ActivationLayer(tanh, tanh_grad))
# CNNnet.add(ActivationLayer(sigmoid, sigmoid_grad))

### Training!

There is no need to train the model on the entire dataset (you'l see why now...). Try to prove that your network works as expected by showing that your error decreases dramatically even on a small sample (say, around 1k).

In [20]:
# While developing, it is recommended to train your model on a subset of the data... / or low epochs.
# Training will be pretty slow if we update at each iteration on 60000 samples...]
CNNnet.use_loss(mse, mse_grad)
epoch_num = 5  # 5 epochs on 1k samples are enough for >85% (see evaluation)
lr = 0.1
t1 = time.time()
CNNnet.fit(x_train[:1000], y_train[:1000], epochs=epoch_num,
           learning_rate=lr)
print(f"Total process time: {round(time.time() - t1,3)}")


Training on 1,000 samples:


Training epoch 1/5   error=0.062085


Training epoch 2/5   error=0.041420


Training epoch 3/5   error=0.036258


Training epoch 4/5   error=0.032845


Training epoch 5/5   error=0.029879
Total process time: 450.827


Hurray - if you got to this point, your CNN model works, but is EXTREMELY slow.   
Let's check why using a cProfiler:

In [21]:
import copy
# profile a copy so that the trained model used for evaluation is left untouched
copy.deepcopy(CNNnet).prof(x_train[:100], y_train[:100], epochs=1, learning_rate=lr)

Training on 100 samples:


Training epoch 1/1   error=0.026134
         27145667 function calls (27145665 primitive calls) in 18.059 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      100    0.589    0.006    2.840    0.028 1102865481.py:20(forward_propagation)
      100    1.849    0.018   11.184    0.112 1102865481.py:36(backward_propagation)
  1382600    0.555    0.000    0.555    0.000 1102865481.py:9(patch_generator)
      100    0.000    0.000    0.002    0.000 2086474222.py:5(forward_propagation)
      100    0.001    0.000    0.002    0.000 2086474222.py:8(backward_propagation)
   135400    0.067    0.000    0.067    0.000 3506474812.py:21(patch_generator)
      100    0.101    0.001    1.118    0.011 3506474812.py:33(forward_propagation)
      100    0.410    0.004    1.461    0.015 3506474812.py:61(backward_propagation)
        2    0.000    0.000    0.000    0.000 <frozen abc>:117(__instancecheck__)
        4    0.000    0.000    0.000 

P2Q2c:  (5pts).     
Why do you think our CNN implementation is slow? Your answer should relate to the  approximate `O(n)` time/computational complexity .

**Answer (P2Q2c).** Every convolution and pooling window is handled in a Python loop, one sample at a time, with no batching. Per sample ($H=28$, $k=3$, $F=12$ filters, pool $p=3$):

| Step | Python iterations per sample |
|---|---|
| Conv forward | $(H-k+1)^2=676$ |
| Conv backward | 676 |
| MaxPool forward | $F\,(26-p+1)^2=6{,}912$ |
| MaxPool backward | 6,912 (each with a `nanargmax` call) |

That's about 15,000 Python iterations and about 270,000 function calls per sample (the profiler shows 27.1 M calls for 100 samples). Each numpy call on a 3×3 window costs microseconds of overhead to do about 10 flops, so the time is dominated by overhead, not by the computation. The cost grows like $O(E\cdot N\cdot F\cdot H^2)$ Python calls, while the actual work could be done with a few vectorized matrix products. Measured: about 90 ms per sample, 451 s for 5 epochs on 1,000 samples. Ten epochs on the 60,000 images would take about 15 hours.

The profiler agrees: `MaxPool.backward_propagation` takes 11.2 s of the 18 s, mostly the 691,200 calls to `nanargmax`. The first affine layer has 885k weights but is cheap, because it is a single numpy call.

To fix it: write the convolution as one matrix product (im2col or `sliding_window_view` + `einsum`), vectorize the pooling and save the argmax in the forward pass, and train in mini-batches.

### Evaluation

Exciting! Now is the time to test your model.     


    May the gradients be always in your favor.

P2Q2b (cont'd):

**Try to limit the number of epochs and the size of the trained data, while still achieving an accuracy score of >85%.**

In [22]:
output = CNNnet.predict(x_test[:500] ,y_test[:500] )


Test set: Avg. loss: 0.0380, Accuracy: 448/500 (90%)



### Benchmarking against PyTorch

How well your model performs against a similar-architecture PyTorch model?   
It is time to find out:

In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset

**Prepare the data as tensors using PyTorch DataLoader:**

In [24]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()
# # training data : 60000 samples
# # reshape and normalize input data
# x_train = x_train.reshape(x_train.shape[0], 1, 28*28) not needed
x_train = x_train.astype('float32')
x_train /= 255
# # One-hot encoding of the output.
# # Currently a number in range [0,9]; Change into a vector of size 10
# # e.g. number 3 will become [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
y_train = to_categorical(y_train)
# # same for test data : 10000 samples
# x_test = x_test.reshape(x_test.shape[0], 1, 28*28) not needed
x_test = x_test.astype('float32')
x_test /= 255
y_test = to_categorical(y_test)

In [25]:
t_train =  TensorDataset(torch.Tensor(x_train),torch.Tensor(y_train))
t_test =  TensorDataset(torch.Tensor(x_test),torch.Tensor(y_test))
train_loader = torch.utils.data.DataLoader(dataset=t_train, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=t_test, batch_size=64, shuffle=False)

Define a `PyTorchCNN` class with an identical architecture you used in your home-made network.  
Hint: use `x = x.view()` to handle reshapes.

P2Q3a: (5pts).   

In [26]:
class PyTorchCNN(nn.Module):
    def __init__(self):
        super(PyTorchCNN, self).__init__()
        input_size = 28 * 28
        num_classes = 10
        self.side = int(np.sqrt(input_size))                 # 28
        self.conv = nn.Conv2d(1, 12, kernel_size=3)          # (1,28,28) -> (12,26,26), stride 1, no padding
        self.pool = nn.MaxPool2d(kernel_size=3, stride=1)    # (12,26,26) -> (12,24,24), stride 1 as in MaxPool
        self.flat = 12 * 24 * 24                             # 6912
        self.fc1 = nn.Linear(self.flat, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = x.view(-1, 1, self.side, self.side)   # (B,28,28) -> (B,1,28,28)
        x = F.relu(self.conv(x))                  # conv + ReLU (as in our Convolution layer)
        x = self.pool(x)
        x = x.view(-1, self.flat)                 # Reshape layer
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        x = torch.tanh(self.fc3(x))
        return x

In [27]:

# Train the model
torch.manual_seed(0)
t_pt = time.time()
num_epochs = 10
pt_learning_rate = 0.01
pt_network = PyTorchCNN()
optimizer = torch.optim.Adam(pt_network.parameters(), lr=pt_learning_rate)
criterion = nn.MSELoss()

for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        # Forward pass
        # labels = labels.view(64,1,10)
        outputs = pt_network(images)
        loss = criterion(outputs, labels)
        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # A handy printout:
        if (i + 1) % 500 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}')

print(f"Total process time: {round(time.time() - t_pt,3)}")

Epoch [1/10], Step [500/938], Loss: 0.0083


Epoch [2/10], Step [500/938], Loss: 0.0096


Epoch [3/10], Step [500/938], Loss: 0.0040


Epoch [4/10], Step [500/938], Loss: 0.0077


Epoch [5/10], Step [500/938], Loss: 0.0066


Epoch [6/10], Step [500/938], Loss: 0.0079


Epoch [7/10], Step [500/938], Loss: 0.0058


Epoch [8/10], Step [500/938], Loss: 0.0124


Epoch [9/10], Step [500/938], Loss: 0.0083


Epoch [10/10], Step [500/938], Loss: 0.0059


Total process time: 124.592


Evaluation:

In [28]:
pt_network.eval()
test_losses = []
test_loss = 0
correct = 0
with torch.no_grad():
    for data, target in test_loader:
        output = pt_network(data)
        test_loss += criterion(output, target,)
        pred = output.data.max(1, keepdim=True)[1]
        correct += pred.eq(target.data.max(1,keepdim=True)[1]).sum()

test_loss /= len(test_loader.dataset)
test_losses.append(test_loss)
print('\nTest set: Avg. loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
  test_loss, correct, len(test_loader.dataset),
  100. * correct / len(test_loader.dataset)))


Test set: Avg. loss: 0.0001, Accuracy: 9667/10000 (97%)



P2Q3b (5pts).   

Time for some questions:
1. Which one of the models performed better? Why?
2. Which one of the models performed faster? Why?  
3. What would you change in your network's architecture?   
4. What would you change in your model's solution algorithm?

Write your solutions here:

| | Home-made numpy CNN | PyTorch CNN (same architecture) |
|---|---|---|
| Training data | 1,000 samples × 5 epochs | 60,000 samples × 10 epochs |
| Optimizer | online SGD, $\eta=0.1$, MSE | Adam, $\eta=0.01$, batch 64, MSE |
| Training time | 450.8 s (about 90 ms per sample) | 124.6 s (about 0.2 ms per sample) |
| Test accuracy | 90 % (448/500) | **96.7 %** (9,667/10,000) |

**1.** PyTorch did better: 96.7 % against 90 %. The architecture is the same (about 894k parameters), so it's not a question of capacity. The difference is the training: PyTorch saw all 60,000 images 10 times (120× more gradient computations) and used Adam (a step size per parameter, plus momentum) instead of plain SGD with one learning rate. The PyTorch "Avg. loss 0.0001" can't be compared with ours: the given evaluation loop sums batch means and then divides by the number of samples, so it is about 64× too small.

**2.** PyTorch was much faster, about 430× per sample (0.2 ms against 90 ms). Its convolution and pooling are compiled C++ kernels working on whole batches of 64, instead of thousands of Python iterations per sample, and the backward pass reuses the max indices saved in the forward pass instead of recomputing them.

**3.** In the architecture I would:
* pool with $2\times2$ windows and stride 2: $12\times13\times13=2{,}028$ features instead of 6,912. `fc1` holds 99 % of the parameters (884,864 of 893,878), so this cuts the parameters by about 3.3×, and the pooling actually downsamples;
* use two conv–ReLU–pool blocks (for example 16 then 32 filters) instead of one;
* use ReLU in the dense layers, and softmax + cross-entropy at the output instead of tanh + MSE;
* add conv biases, batch norm, and dropout before the dense layers.

**4.** In the algorithm I would:
* vectorize the convolution and pooling and train with `fit_mini_batch`, which would make training on the full 60k feasible;
* use momentum or Adam with a learning-rate schedule, and He initialization for the ReLU layers;
* keep a validation set with early stopping, add small data augmentation (shifts, rotations), and shuffle every epoch (`fit` always goes through the samples in the same order).